# **Lecture 1 — Part 1: Training the Roboracer car with Reinforcement Learning** 🏎️

---

**Welcome!** This notebook is intended to act as a standalone training guide for running your RoboRacer car autonomously using Reinforcement Learning. We will implement PPO step by step, starting from a car that crashes immediately, and ending with one that completes laps on its own.

No hand-coded rules. No human demonstrations.

---

## 📋 Before You Start

### What You Need
- A Google account (for Colab)
- **No ROS installation required** — everything runs in this notebook
- **No prior RL experience required** — we explain every concept as we go

### What You Will Learn
- How to frame a robotics problem as an RL problem (observations, actions, rewards)
- How PPO works at an intuitive level
- How to wrap a simulator for RL training
- How to train, monitor, and evaluate a policy

### ⏱️ Runtime
- **Training time: ~25–30 minutes** on a Colab T4 GPU
- Make sure you have selected **Runtime → Change runtime type → T4 GPU** before starting

---

> **📌 Note:** This is Part 1. We deliberately keep the reward function simple — the car will learn to drive but may be a bit squiggly or slow in places. In Part 2 we fix that with reward shaping.

---

## 📝 Student TODOs — Complete These Before Running Training

This notebook has **5 sections** where you must fill in your own implementation.
The rest of the code is provided — but training will not work correctly until
all TODOs are completed.

| # | TODO | Location | Difficulty |
|---|---|---|---|
| **TODO 1** | Define the observation space dimensions and bounds | Section 3 — Wrapper `__init__` | ⭐ |
| **TODO 2** | Normalize the lidar scan in `_process_obs` | Section 3 — Wrapper `_process_obs` | ⭐ |
| **TODO 3** | Scale network actions to physical steering and speed | Section 3 — Wrapper `_scale_action` | ⭐⭐ |
| **TODO 4** | Design the reward function in `step()` | Section 4 — Wrapper `step()` | ⭐⭐⭐ |
| **TODO 5** | Choose PPO hyperparameters | Section 5 — PPO Model | ⭐⭐ |

> **📌 Tip:** Read each section's explanation carefully before filling in its TODO.
> The markdown cells above each code block explain the concept and give you hints.
> You can always refer to the **Section 1 concepts** and **Section 4 reward** explanations
> if you are unsure what values to use.

---


---

# **Section 1 — Concepts**

## 1.1 What Are We Actually Doing?

Before touching any code, let's make sure we understand what Reinforcement Learning means in the context of our race car.

### The RL Loop

At every timestep (100 times per second in our simulator), this loop runs:

```
┌─────────────────────────────────────────────────────┐
│                                                     │
│   Simulator ──► Observation ──► Policy Network      │
│       ▲                              │              │
│       │                              ▼              │
│    Reward ◄────────────────── Action (steer, speed) │
│                                                     │
└─────────────────────────────────────────────────────┘
```

The car sees the world through its sensors (our observation), a neural network decides what to do (our policy), the simulator executes that decision and returns a score (the reward), and the cycle repeats.

Over thousands of laps the network gradually learns which actions lead to higher rewards — without ever being told explicitly how to drive.

### Mapping RL Concepts to Our Car

| RL Concept | What It Means for Our Car |
|---|---|
| **Observation** | 108 lidar beam distances + current speed + current steering angle |
| **Action** | Steering angle and speed command |
| **Reward** | +small for moving forward, −large for crashing |
| **Policy** | A small neural network (2 hidden layers of 64 neurons) |
| **Episode** | One attempt at driving — ends when car crashes or time runs out |

**Why 108 lidar beams?** The car's lidar fires 1080 beams per scan. We keep every 10th beam (108 total) to keep the network small and fast, while still covering the full 270° field of view for this training. We then add the car's current speed and steering angle as a part of the observation space, giving the network the context it needs to decide what to do next. The network never sees the car's global position or the track map. It only ever sees these 110 numbers. We cover this in more detail in Section 3.

### Why PPO?

PPO (Proximal Policy Optimization) is one of the most reliable RL algorithms for continuous control tasks like driving. It works by:

1. Collecting a batch of experience (the car drives around for a while)
2. Computing how much better or worse each action was than expected (the *advantage*)
3. Updating the policy to do more of the good actions and fewer of the bad ones
4. Repeating — but with a *clip* that prevents any single update from changing the policy too drastically

That last point is the "Proximal" part. Without it, a single bad batch of experience can destroy a policy that took millions of steps to learn. The clip prevents that. We will see this as `clip_range=0.2` in the code, it means the policy is never allowed to change by more than 20% in a single update.

**Now, let's build it.**


---

# **Section 2 — Setup**

## 2.1 Install Dependencies

We need three things:

1. **`f1tenth_gym`** — the physics simulator (the "world" our car lives in)
2. **`stable-baselines3`** — the library that provides PPO
3. **`wandb`** — for tracking our training metrics in real time

▶️ Run the cell below. It will take about **2–3 minutes** the first time.

In [ ]:
# ── Install all dependencies ──────────────────────────────────────────────────
# f1tenth_gym: the race car simulator (dev-humble branch = latest stable)
!pip install -q git+https://github.com/f1tenth/f1tenth_gym.git@dev-humble

# stable-baselines3: provides PPO and all the training infrastructure
# [extra] adds progress bars, rich logging, pandas support
!pip install -q stable-baselines3[extra]

# wandb: experiment tracking — we will log metrics here in real time
!pip install -q wandb

print("Installation complete!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.9/953.9 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gym

## 2.2 Verify Imports

In [ ]:
# ── Verify everything imports correctly ───────────────────────────────────────
import gymnasium as gym
import numpy as np
import torch
import stable_baselines3
import wandb
import f1tenth_gym

print(f"PyTorch:          {torch.__version__}")
print(f"CUDA available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:              {torch.cuda.get_device_name(0)}")
print(f"Stable Baselines: {stable_baselines3.__version__}")
print(f"Gymnasium:        {gym.__version__}")
print()
print("All good — let's start training!")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


PyTorch:          2.11.0+cu128
CUDA available:   True
GPU:              Tesla T4
Stable Baselines: 2.9.0
Gymnasium:        0.29.1

All good — let's start training!


## 2.3 Pre-Cache the Track Map

> The first time a track is loaded, the gym downloads and extracts it. If multiple parallel environments try to do this simultaneously they can conflict. We pre-cache it here (once, single-threaded) to avoid that race condition.

Here we are selecting the Spielberg Map, but you can select your own map files to train your car on the appropriate race track.

In [ ]:
# ── Pre-cache the track map ───────────────────────────────────────────────────
from f1tenth_gym.envs.track import Track
Track.from_track_name('Spielberg')
print("Spielberg track cached — ready to train!")

Extracting Files for: Spielberg
Spielberg track cached — ready to train!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---

# **Section 3 — The Environment**

## 3.1 The Simulator Wrapper

The F1TENTH gym returns observations as a nested dictionary:

```python
{
  "agent_0": {
    "scan":      [...],   # 1080 lidar beams (distances in metres)
    "std_state": [...],   # [x, y, steering_angle, speed, heading, yaw_rate, slip_angle]
    "collision": False    # True if car is touching a wall this step
  }
}
```

RL algorithms like PPO expect a **flat numpy array**, a single list of numbers, not a nested dictionary. Our `F1TenthRLWrapper` class bridges this gap. Think of it as a translator sitting between the simulator and the learning algorithm.

### What the Wrapper Does

| Step | What Happens | Why |
|---|---|---|
| **1. Extract** | Pulls lidar scan and `std_state` from the nested dict | PPO can't read nested dicts, it needs a flat array |
| **2. Downsample lidar** | 1080 beams → 108 beams (every 10th reading) | Smaller input = smaller faster network. 108 beams still gives full 270° coverage, enough to detect walls, gaps, and corners |
| **3. Normalize** | All values scaled to [0, 1] | Neural networks train much faster when all inputs are on the same scale. |
| **4. Add speed + steer** | Appends 2 numbers to obs → 110-dim total | The car gets context about how fast it's actually going and how much it's already steering to make better informed decisions. |
| **5. Scale actions** | Network outputs [-1, 1] → physical steer (rad) + speed (m/s) | The network always outputs numbers in [-1, 1]. We map those to physical limits: steering ±0.4189 rad, speed 1.5–6.0 m/s |
| **6. Compute reward** | Calculates the reward signal each step | This is the score the car gets each step, it drives the learning. Covered in Section 4. |

### What the Observation Looks Like

```
obs = [lidar_0, lidar_1, ..., lidar_107,  speed_norm,  steer_norm]
       ←-------- 108 values -----------►  ←--- 1 ---►  ←--- 1 ---►
       all normalized to [0, 1]                        total: 110

lidar value = 0.0  → wall is right there (0 m away)
lidar value = 1.0  → open space, nothing within 10 m
speed value = 1.0  → car is at max speed (6 m/s)
speed value = 0.0  → car is stationary
```

> **Note:** `std_state[0]` and `std_state[1]` are the car's global x, y position. We use these only to compute the progress reward — they are **never part of the observation the network sees**. The policy works purely from sensor readings, which means it runs identically on the real car without any localization or GPS.

Read through the class below, every important line has a comment explaining *why* it is there.

*(This is not the training block, hence you can just glance over this wrapper.)*


In [ ]:
import gymnasium as gym
import numpy as np


class F1TenthRLWrapper(gym.Env):
    """
    Wraps the F1TENTH gym into a standard RL environment.

    Observation (110-dim, all values in [0, 1]):
        [0:108]  108 lidar beam distances, normalized to [0, 1]
        [108]    current speed, normalized by max_speed
        [109]    current steering angle, normalized to [0, 1]

    Action (2-dim, values in [-1, 1]):
        [0]  steering command  → scaled to gym physical limits (rad)
        [1]  speed command     → scaled to [min_speed, max_speed] m/s

    Reward (Part 1 — simple):
        +delta_progress * 10   forward progress along the track
        +0.03 * (speed/vmax)   small speed bonus (prevents crawling)
        -30.0                  crash penalty (episode ends)
    """

    def __init__(self, map_name="Spielberg", num_beams=108, render_mode=None):
        super().__init__()

        import f1tenth_gym
        from f1tenth_gym.envs.env_config import (
            EnvConfig, SimulationConfig, ObservationConfig, ResetConfig,
        )
        from f1tenth_gym.envs.lidar.config import LiDARConfig
        from f1tenth_gym.envs.observation import ObservationType
        from f1tenth_gym.envs.reset import ResetStrategy
        from f1tenth_gym.envs.integrators import IntegratorType
        from f1tenth_gym.envs.dynamic_models import DynamicModel

        env_config = EnvConfig(
            map_name=map_name,
            num_agents=1,
            simulation_config=SimulationConfig(
                timestep=0.01,
                integrator=IntegratorType.RK4,
                dynamics_model=DynamicModel.ST,
                max_laps=5,
            ),
            observation_config=ObservationConfig(type=ObservationType.DIRECT),
            lidar_config=LiDARConfig(
                num_beams=num_beams,
                range_max=10.0,
                noise_std=0.0,
            ),
            reset_config=ResetConfig(strategy=ResetStrategy.RL_RANDOM_STATIC),
            render_enabled=(render_mode is not None),
        )

        self.env = gym.make(
            "f1tenth_gym:f1tenth-v0",
            config=env_config,
            render_mode=render_mode,
        )

        self.max_steer  = float(self.env.action_space.high[0][0])
        self.max_speed  = 6.0
        self.min_speed  = 1.5
        self.lidar_clip = 10.0
        self.num_beams  = num_beams
        self.max_steps  = 10000
        self.current_step = 0

        try:
            self.track_length = self.env.unwrapped.track.centerline.length
        except Exception:
            self.track_length = None

        self.waypoints      = self._extract_waypoints()
        self.prev_progress  = 0.0
        self.start_progress = 0.0
        self.prev_lap_count = 0

        print(f"[ENV] max_steer={self.max_steer:.4f}rad | max_speed={self.max_speed:.1f}m/s")

        # ═══════════════════════════════════════════════════════════════
        # ✏️  TODO 1 — Define the Observation Space
        # ═══════════════════════════════════════════════════════════════
        # The observation is a flat vector of numbers the policy network sees.
        # You need to tell SB3 what shape it is and what range the values fall in.
        #
        # Our observation has:
        #   - num_beams lidar readings  (each normalized to [0, 1])
        #   - 1 speed value             (normalized to [0, 1])
        #   - 1 steering angle value    (normalized to [0, 1])
        #
        # Use gym.spaces.Box to define the space.
        # Hint: low=0.0 everywhere, high=1.0 everywhere (all values are normalized)
        # Hint: total dimensionality = num_beams + 2
        #
        # Reference: https://gymnasium.farama.org/api/spaces/fundamental/#gymnasium.spaces.Box
        # ───────────────────────────────────────────────────────────────
        self.observation_space = gym.spaces.Box(
            low   = ???,   # minimum value for each dimension (scalar or array)
            high  = ???,   # maximum value for each dimension (scalar or array)
            shape = ???,   # tuple: total number of dimensions
            dtype = np.float32
        )
        # ═══════════════════════════════════════════════════════════════

        self.action_space = gym.spaces.Box(
            low=-1.0, high=1.0, shape=(2,), dtype=np.float32
        )

    # ─────────────────────────────────────────────────────────────────────────

    def _extract_waypoints(self):
        """Get the track centerline as an (N, 2) array of [x, y] coordinates."""
        try:
            track = self.env.unwrapped.track
            line  = getattr(track, "raceline", None) or getattr(track, "centerline", None)
            if line is not None:
                wps = np.column_stack([line.xs, line.ys])
                print(f"[ENV] Loaded {len(wps)} waypoints from track")
                return wps
        except Exception as e:
            print(f"[ENV] WARNING: Could not load waypoints: {e}")
        print("[ENV] WARNING: Using dummy circle waypoints — reward will be incorrect!")
        t = np.linspace(0, 2 * np.pi, 100, endpoint=False)
        return np.column_stack([5 * np.cos(t), 5 * np.sin(t)])

    def _get_progress(self, raw_obs):
        """
        Returns progress 0.0 to 1.0 along the track.
        Primary: gym frenet s / track_length. Fallback: waypoint search.
        """
        try:
            frenet = raw_obs["agent_0"].get("frenet_pose", None)
            if frenet is not None and self.track_length:
                s = float(frenet[0])
                return (s % self.track_length) / self.track_length
        except Exception:
            pass
        state   = raw_obs["agent_0"]["std_state"]
        x, y    = float(state[0]), float(state[1])
        dists   = np.linalg.norm(self.waypoints - np.array([x, y]), axis=1)
        closest = np.argmin(dists)
        return closest / len(self.waypoints)

    def _process_obs(self, raw_obs):
        """
        Converts the gym's nested dict observation into a flat (110,) array.
        The gym already fired exactly 108 beams and clipped them to 10m via LiDARConfig.
        Your job here is only to normalize the values to [0, 1].
        """
        agent = raw_obs["agent_0"]
        scan  = agent["scan"]  # shape: (108,) — already clipped to 10m by LiDARConfig

        # ═══════════════════════════════════════════════════════════════
        # ✏️  TODO 2 — Normalize the Lidar Scan
        # ═══════════════════════════════════════════════════════════════
        # The scan values are in metres (0m to 10m).
        # Neural networks train best when all inputs are in [0, 1].
        # Divide by the maximum range so 10m → 1.0, 0m → 0.0.
        #
        # Hint: self.lidar_clip holds the maximum range value (10.0)
        # Hint: convert to float32 after dividing
        # ───────────────────────────────────────────────────────────────
        scan = ???   # normalize scan to [0, 1]
        # ═══════════════════════════════════════════════════════════════

        state      = agent["std_state"]
        speed_norm = np.clip(float(state[3]) / self.max_speed, 0.0, 1.0)
        steer_norm = np.clip(
            (float(state[2]) + self.max_steer) / (2 * self.max_steer), 0.0, 1.0
        )

        return np.concatenate([
            scan.astype(np.float32),
            np.array([speed_norm, steer_norm], dtype=np.float32)
        ])

    def _scale_action(self, action):
        """
        Converts network output in [-1, 1] to physical commands.
        The network always outputs values between -1 and 1.
        We must map these to real physical units the car understands.
        """
        # ═══════════════════════════════════════════════════════════════
        # ✏️  TODO 3 — Scale Network Actions to Physical Commands
        # ═══════════════════════════════════════════════════════════════
        # action[0] is the raw steering command in [-1, 1]
        # action[1] is the raw speed command in [-1, 1]
        #
        # STEERING:
        #   Map [-1, 1] → [-max_steer, +max_steer] radians
        #   Hint: simple multiplication by self.max_steer
        #   e.g. action[0]=1.0  → full right (+0.4189 rad)
        #        action[0]=-1.0 → full left  (-0.4189 rad)
        #        action[0]=0.0  → straight   (0.0 rad)
        #
        # SPEED:
        #   Map [-1, 1] → [min_speed, max_speed] m/s
        #   Hint: use the formula that maps [-1,1] → [a, b]:
        #         value = (x + 1) / 2 * (b - a) + a
        #   e.g. action[1]=-1.0 → 1.5 m/s (slowest)
        #        action[1]=+1.0 → 6.0 m/s (fastest)
        #        action[1]=0.0  → 3.75 m/s (midpoint)
        #
        # Return shape must be (1, 2) — the gym expects a batched action
        # ───────────────────────────────────────────────────────────────
        steer = ???   # scale action[0] to steering radians
        speed = ???   # scale action[1] to m/s

        return np.array([[steer, speed]], dtype=np.float32)
        # ═══════════════════════════════════════════════════════════════

    # ─────────────────────────────────────────────────────────────────────────

    def reset(self, seed=None, options=None):
        raw_obs, info     = self.env.reset(seed=seed)
        self.current_step = 0
        self.prev_lap_count = 0

        self.start_progress = self._get_progress(raw_obs)
        self.prev_progress  = self.start_progress

        return self._process_obs(raw_obs), info

    def step(self, action):
        physical_action = self._scale_action(action)
        raw_obs, _, done, truncated, info = self.env.step(physical_action)
        self.current_step += 1

        state     = raw_obs["agent_0"]["std_state"]
        speed     = float(state[3])
        collision = bool(raw_obs["agent_0"].get("collision", False))
        scan_raw  = raw_obs["agent_0"]["scan"]

        lap_count     = int(raw_obs["agent_0"].get("lap_count", 0))
        lap_completed = lap_count > self.prev_lap_count
        self.prev_lap_count = lap_count

        # Progress along the track (0.0 → 1.0)
        progress       = self._get_progress(raw_obs)
        delta_progress = progress - self.prev_progress
        if delta_progress < -0.5:
            delta_progress += 1.0

        # ═══════════════════════════════════════════════════════════════
        # ✏️  TODO 4 — Design the Reward Function
        # ═══════════════════════════════════════════════════════════════
        # This is the most important part of RL — the reward signal tells
        # the agent what behavior is good and what is bad.
        # The agent will optimize EXACTLY what you reward, nothing more.
        #
        # You have access to:
        #   delta_progress  — how much further along the track this step (small float ~0.001)
        #   speed           — current car speed in m/s (0.0 to 6.0)
        #   self.max_speed  — maximum speed (6.0 m/s)
        #   collision       — True if the car hit a wall this step
        #
        # REWARD COMPONENT 1 — Progress reward (required)
        #   The car should be rewarded for moving forward along the track.
        #   delta_progress is very small (~0.001 per step), so scale it up.
        #   Hint: multiply by a value between 5 and 20 to get meaningful reward per step
        #
        # REWARD COMPONENT 2 — Speed bonus (recommended)
        #   Without this the car may learn to drive very slowly to avoid crashes.
        #   Keep this SMALL — if it is too large the car chases speed over progress.
        #   Hint: 0.03 * (speed / self.max_speed) gives max +0.03 per step
        #
        # REWARD COMPONENT 3 — Crash penalty (required)
        #   When collision=True the car has hit a wall. Give it a large negative reward
        #   and end the episode (set done=True).
        #   Hint: the penalty should be large enough to outweigh many steps of progress
        #         A good starting point is -30.0
        #   Important: when collision=True, override ALL other reward components
        #              with just the penalty (don't add progress + penalty together)
        # ───────────────────────────────────────────────────────────────
        reward = 0.0

        # Component 1: Progress reward
        reward += ???   # reward the car for moving forward

        # Component 2: Speed bonus
        reward += ???   # small bonus for going faster (prevent crawling)

        # Component 3: Crash penalty
        if collision:
            reward = ???   # large negative reward — overrides everything above
            done   = True  # end the episode immediately
        # ═══════════════════════════════════════════════════════════════

        if self.current_step >= self.max_steps:
            truncated = True

        distance_traveled = progress - self.start_progress
        if distance_traveled < 0:
            distance_traveled += 1.0

        info["speed"]         = speed
        info["collision"]     = collision
        info["progress"]      = distance_traveled
        info["min_scan"]      = float(np.min(scan_raw))
        info["lap_completed"] = lap_completed

        self.prev_progress = progress
        return self._process_obs(raw_obs), float(reward), done, truncated, info

    def render(self):
        return self.env.render()

    def close(self):
        self.env.close()


print("F1TenthRLWrapper defined successfully.")
print("Remember: complete all TODOs before running training!")


F1TenthRLWrapper defined successfully.


## 3.2 Inspect the Environment

Before training, let's create one environment instance and look at what the car actually sees.

In [ ]:
# ── Instantiate one environment and look inside ───────────────────────────────
env = F1TenthRLWrapper(map_name="Spielberg")
obs, info = env.reset(seed=42)

print("─" * 50)
print("Observation space:", env.observation_space)
print("Action space:     ", env.action_space)
print()
print("Observation shape:", obs.shape)   # should be (110,)
print("Observation min:  ", obs.min().round(3))
print("Observation max:  ", obs.max().round(3))
print()
print("First 5 lidar beams (normalized 0–1):", obs[:5].round(3))
print("Speed (normalized):                  ", obs[108].round(3))
print("Steering (normalized):               ", obs[109].round(3))
print("─" * 50)

# Take one random action to see what a step looks like
random_action = env.action_space.sample()
obs, reward, terminated, truncated, info = env.step(random_action)
print()
print("After one random action:")
print(f"  Action taken:  steer={random_action[0]:.3f}, speed_cmd={random_action[1]:.3f}")
print(f"  Reward:        {reward:.4f}")
print(f"  Speed:         {info['speed']:.2f} m/s")
print(f"  Min lidar:     {info['min_scan']:.2f} m")
print(f"  Progress:      {info['progress']:.4f} (fraction of lap)")
print(f"  Collision:     {info['collision']}")

env.close()

[ENV] Loaded 1692 waypoints from track
[ENV] max_steer=0.4189rad | max_speed=20.0m/s


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


──────────────────────────────────────────────────
Observation space: Box(0.0, 1.0, (110,), float32)
Action space:      Box(-1.0, 1.0, (2,), float32)

Observation shape: (110,)
Observation min:   0.0
Observation max:   0.5

First 5 lidar beams (normalized 0–1): [0. 0. 0. 0. 0.]
Speed (normalized):                   0.0
Steering (normalized):                0.5
──────────────────────────────────────────────────


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag


After one random action:
  Action taken:  steer=0.991, speed_cmd=0.581
  Reward:        0.0002
  Speed:         0.10 m/s
  Min lidar:     0.44 m
  Progress:      0.0000 (fraction of lap)
  Collision:     False


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
raw_obs, _ = env.env.reset()
frenet = raw_obs["agent_0"].get("frenet_pose", None)
print("frenet_pose:", frenet)
print("track_length:", env.track_length)

frenet_pose: [224.88432     -0.30908212  -0.25153422]
track_length: 343.3226318359375


---

# **Section 4 — The Reward Function**

## 4.1 Part 1 Reward Design

The reward function is the most important design decision in RL. **The agent will optimize exactly what you reward — nothing more, nothing less.**

Our Part 1 reward is intentionally simple — just two questions per step:

```
Did you move forward?   → reward = delta_progress × 10
Are you going fast?     → reward += 0.03 × (speed / vmax)
Did you crash?          → reward = −30.0, episode over
```

### Why These Scales?

| Term | Typical Value per Step | Purpose |
|---|---|---|
| `delta_progress × 10` | +0.01 to +0.05 | Main signal — move forward along the track |
| `0.03 × (v/vmax)` | +0.005 to +0.03 | Prevent the car learning to crawl slowly |
| Crash penalty | −30.0 (once, terminal) | Strong signal to avoid walls |

The speed bonus is kept **much smaller** than the progress signal. If it were too large, the car would learn to drive fast into walls — high speed reward minus one crash penalty might still be net positive. The relative scale between reward terms matters as much as the terms themselves.

`You can also try training without the small speed reward to start with.`

### What Will Go Wrong?

With this simple reward, the car will:
- ✅ Learn to drive forward without crashing (most of the time)
- ⚠️ Drive with a slightly wobbly steering pattern — because we never penalized jerky inputs
- ⚠️ Hug walls on corners — because getting close to a wall has no cost, only hitting it does
- ⚠️ No special signal for completing a full lap — same reward whether it finishes or stops at 99%

We will fix all of these in Part 2 with reward shaping. For now, let's get a simple version running.

> 💡 **Try this after training:** Change `crash_penalty = -5.0` and retrain. What happens to the driving behavior? Does the car take more risks?


---

# **Section 5 — PPO Training Setup**

## 5.1 Training Pipeline Overview

Now let's set up the full training pipeline. There are a few layers here — let's understand each one before we run.

### The Environment Wrapper Stack

```
SubprocVecEnv          ← runs 4 environments in parallel (4× more data/second)
    └── VecMonitor     ← logs episode rewards and lengths automatically
        └── VecNormalize  ← normalizes rewards for stable training
```

> **Why VecNormalize?** Raw rewards range from -30 (crash) to +0.05 (one good step). This huge range makes gradient updates unstable. VecNormalize tracks a running mean and std of rewards and scales them to a consistent range — similar to how BatchNorm works in supervised learning.

> **⚠️ Important:** VecNormalize **must be the outermost wrapper** — SB3's EvalCallback walks the stack from the outside looking for it.


▶️ **How PPO Works — The Core Math**

At each update step PPO computes how much the new policy differs from the old one, then clips it to prevent large changes. This can simply be understood as a standard gradient ascent, where we nudge the network weights in the direction that increases expected reward, but with the clip acting as a safety rail that limits how far each nudge can go.

**Step 1 — Compute the ratio:**

> `ratio = π_new(action) / π_old(action)`  
> How much more or less likely is this action under the new policy vs the old one.

**Step 2 — Clip and take the conservative option:**

> `L_CLIP = mean( min( ratio × advantage,  clip(ratio, 1−ε, 1+ε) × advantage ) )`  
> where `ε = clip_range = 0.2`

What it means:
- Action was **good** (positive advantage) → increase its probability, but **not by more than 20%**
- Action was **bad** (negative advantage) → decrease its probability, but **not by more than 20%**

The `min()` ensures PPO always takes the more conservative option — it can never over-commit to what looks like a good update based on a lucky batch.

**What is advantage?**

> `advantage = actual_return − predicted_value`

How much better or worse was this action than the value function expected? Positive → do more of this. Negative → do less. The value function is a second network head that learns to predict expected returns, making advantage estimates more accurate over time.



### PPO Hyperparameters

| Parameter | Value | What It Controls |
|---|---|---|
| `learning_rate` | 3e-4 | How fast network weights update |
| `n_steps` | 4096 | Steps collected before each update |
| `batch_size` | 256 | Minibatch size for gradient descent |
| `n_epochs` | 10 | Passes over collected data per update |
| `gamma` | 0.99 | Discount — how much future rewards matter |
| `gae_lambda` | 0.95 | Advantage smoothing (bias/variance tradeoff) |
| `clip_range` | 0.2 | PPO clip — limits policy change per update |
| `ent_coef` | 0.001 | Entropy bonus — encourages exploration |

The network is a simple MLP: **110 inputs → 64 → 64 → 2 outputs**

## 5.2 Imports

In [ ]:
import os
import wandb
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import (
    SubprocVecEnv, DummyVecEnv, VecMonitor, VecNormalize
)
from stable_baselines3.common.callbacks import (
    BaseCallback, EvalCallback, CheckpointCallback
)

print("Modules imported successfully!")

Modules imported successfully!


## 5.3 Training Configuration

> 🔧 **IMPORTANT - These are the values you can change to experiment.**

In [ ]:
# ── Training configuration ────────────────────────────────────────────────────
# Change these to experiment!

MAP_NAME    = "Spielberg"   # Track to train on
NUM_ENVS    = 4             # Parallel environments (reduce to 2 if Colab is slow)
TOTAL_STEPS = 1_000_000       # Total training steps (~30 min on T4 GPU)
DEVICE      = "cuda"        # Use GPU — much faster for rollout collection

print(f"Training config:")
print(f"  Map:    {MAP_NAME}")
print(f"  Envs:   {NUM_ENVS}")
print(f"  Steps:  {TOTAL_STEPS:,}")
print(f"  Device: {DEVICE}")

Training config:
  Map:    Spielberg
  Envs:   4
  Steps:  1,000,000
  Device: cuda


## 5.4 WandB Login

WandB tracks all our metrics in real time. You will need a free account at [wandb.ai](https://wandb.ai) — it takes 30 seconds to sign up.

After running this cell, click the link it prints to see your live dashboard.

> **Get your API key at:** [wandb.ai/authorize](https://wandb.ai/authorize)

In [ ]:
# ── WandB login ───────────────────────────────────────────────────────────────
import wandb

# Choose option 1 or 2, then paste your WANDB_API key
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
/usr/local/lib/python3.12/dist-packages/notebook/utils.py:280: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  return LooseVersion(v) >= LooseVersion(check)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results



KeyboardInterrupt: 

## 5.5 Metrics Callback

SB3 logs basic training metrics automatically. This callback adds **racing-specific metrics** that tell us how the car is actually performing on the track. *This will help you analyze your training.*

In [ ]:
# ── Metrics Callback ──────────────────────────────────────────────────────────
class RacingMetricsCallback(BaseCallback):
    """
    Logs per-episode racing metrics to WandB and TensorBoard:

    racing/avg_speed           — average speed during episode (m/s)
    racing/collision_rate      — fraction of episodes ending in crash
    racing/track_progress      — distance traveled (0→1 = one full lap)
    racing/avg_min_scan        — average closest wall distance (safety proxy)
    racing/lap_completion_rate — fraction of episodes completing a full lap
    rollout/ep_len_std         — std of episode lengths (consistency measure)
    perf/fps_per_env           — simulator steps per second per environment
    """

    def __init__(self, num_envs=1, verbose=0):
        super().__init__(verbose)
        self.num_envs      = num_envs
        self.ep_speeds     = []
        self.ep_min_scans  = []
        self.ep_lengths    = []
        self.ep_progress   = 0.0
        self.ep_collisions = 0
        self.ep_laps       = 0
        self.ep_count      = 0
        self.ep_len_buf    = 0

    def _on_step(self) -> bool:
        infos = self.locals.get("infos", [])
        dones = self.locals.get("dones", [])

        # Get fps from SB3 logger
        fps = 0
        try:
            fps = self.model.logger.name_to_value.get("time/fps", 0)
        except Exception:
            pass

        for info, done in zip(infos, dones):
            self.ep_len_buf += 1
            if "speed"    in info: self.ep_speeds.append(info["speed"])
            if "min_scan" in info: self.ep_min_scans.append(info["min_scan"])
            if "progress" in info: self.ep_progress = info["progress"]
            if info.get("collision",     False): self.ep_collisions += 1
            if info.get("lap_completed", False): self.ep_laps       += 1

            if done and len(self.ep_speeds) > 0:
                self.ep_count += 1
                self.ep_lengths.append(self.ep_len_buf)
                self.ep_len_buf = 0

                avg_speed           = float(np.mean(self.ep_speeds))
                avg_min_scan        = float(np.mean(self.ep_min_scans))
                collision_rate      = self.ep_collisions / self.ep_count
                lap_completion_rate = self.ep_laps / self.ep_count
                ep_len_std          = float(np.std(self.ep_lengths)) if len(self.ep_lengths) > 1 else 0.0
                fps_per_env         = fps / max(1, self.num_envs)

                # Log to TensorBoard
                self.logger.record("racing/avg_speed",           avg_speed)
                self.logger.record("racing/avg_min_scan",        avg_min_scan)
                self.logger.record("racing/collision_rate",       collision_rate)
                self.logger.record("racing/track_progress",       self.ep_progress)
                self.logger.record("racing/lap_completion_rate",  lap_completion_rate)
                self.logger.record("rollout/ep_len_std",          ep_len_std)
                self.logger.record("perf/fps_per_env",            fps_per_env)

                # Log to WandB
                wandb.log({
                    "racing/avg_speed":          avg_speed,
                    "racing/avg_min_scan":       avg_min_scan,
                    "racing/collision_rate":      collision_rate,
                    "racing/track_progress":      self.ep_progress,
                    "racing/lap_completion_rate": lap_completion_rate,
                    "rollout/ep_len_std":         ep_len_std,
                    "perf/fps_per_env":           fps_per_env,
                    "global_step":                self.num_timesteps,
                })

                self.ep_speeds    = []
                self.ep_min_scans = []

        return True


print("RacingMetricsCallback defined successfully!")

## 5.6 Environment Builders

In [ ]:
# ── Environment factory ───────────────────────────────────────────────────────
# SubprocVecEnv needs a list of functions that each create one environment.
# This factory pattern is the standard SB3 approach.

def make_env(map_name, rank, seed=0):
    """Returns a function that creates one F1TenthRLWrapper instance."""
    def _init():
        env = F1TenthRLWrapper(map_name=map_name)
        env.reset(seed=seed + rank)   # different seed per env = diverse experience
        return env
    return _init


def make_training_env(map_name, num_envs):
    """
    Creates the vectorized training environment.
    Wrapper order matters — VecNormalize MUST be outermost.
    """
    if num_envs == 1:
        env = DummyVecEnv([make_env(map_name, 0)])       # single env, no subprocess
    else:
        env = SubprocVecEnv([make_env(map_name, i) for i in range(num_envs)])
    env = VecMonitor(env)      # adds episode reward/length logging
    env = VecNormalize(        # normalizes rewards — outermost!
        env,
        norm_obs    = False,   # lidar already in [0,1] — don't normalize again
        norm_reward = True,    # normalize rewards for stable gradients
        clip_reward = 10.0,
        gamma       = 0.99,
    )
    return env


def make_eval_env(map_name):
    """
    Creates the evaluation environment.
    Same wrapper order, but training=False freezes normalization stats.
    """
    env = DummyVecEnv([make_env(map_name, 99)])
    env = VecMonitor(env)
    env = VecNormalize(
        env,
        norm_obs    = False,
        norm_reward = True,
        clip_reward = 10.0,
        gamma       = 0.99,
        training    = False,   # freeze stats — don't update during eval
    )
    return env


print("Environment builders defined successfully.")

---

# **Section 6 — Training**

## 6.1 Launch Training

*Everything is in place. Let's train.*

### What to Expect

| Step Count | What You Should See |
|---|---|
| First few thousand steps | Car crashes constantly, reward is very negative |
| ~100k steps | Collision rate starts dropping, car survives longer |
| ~300k steps | Car completes partial laps regularly |
| ~500k steps | Car completes full laps most of the time |

Watch the WandB link that prints below — your metrics update in real time.

In [ ]:
# ── Initialize WandB run ──────────────────────────────────────────────────────
run = wandb.init(
    project          = "f1tenth-rl-lecture1",
    name             = f"ppo_part1_{MAP_NAME}_{TOTAL_STEPS//1000}k",
    config           = {
        # Environment
        "map":            MAP_NAME,
        "num_envs":       NUM_ENVS,
        "obs_dim":        110,
        "max_speed":      6.0,
        "min_speed":      1.5,
        "max_steps_ep":   10000,
        # Reward
        "progress_scale": 10.0,
        "speed_reward":   0.03,
        "crash_penalty":  -30.0,
        # PPO
        "total_steps":    TOTAL_STEPS,
        "lr":             3e-4,
        "n_steps":        4096,
        "batch_size":     256,
        "n_epochs":       10,
        "gamma":          0.99,
        "gae_lambda":     0.95,
        "clip_range":     0.2,
        "ent_coef":       0.001,
        "net_arch":       "[64, 64] Tanh",
    },
    sync_tensorboard = True,
    monitor_gym      = True,
)

print(f"WandB run: {run.url}")
print("Open the link above to watch your training metrics in real time!")

/usr/local/lib/python3.12/dist-packages/wandb/analytics/sentry.py:269: DeprecationWarning: Read the `app_url` setting from the appropriate Settings object.
  app_url = wandb.util.app_url(tags["base_url"])  # type: ignore[index]
/usr/local/lib/python3.12/dist-packages/wandb/analytics/sentry.py:280: DeprecationWarning: The `Scope.user` setter is deprecated in favor of `Scope.set_user()`.
  self.scope.user = {"email": email}


WandB run: https://wandb.ai/kvirmani-university-of-pennsylvania/f1tenth-rl-lecture1/runs/zr1hkghy
Open the link above to watch your training metrics in real time!


/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())


In [ ]:
# ── Create environments ───────────────────────────────────────────────────────
print("Creating training environment...")
train_env = make_training_env(MAP_NAME, NUM_ENVS)

print("Creating eval environment...")
eval_env = make_eval_env(MAP_NAME)

print(f"Training env obs shape: {train_env.observation_space.shape}")
print(f"Training env act shape: {train_env.action_space.shape}")

Creating training environment...
Creating eval environment...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())


[ENV] Loaded 1692 waypoints from track
Training env obs shape: (110,)
Training env act shape: (2,)


In [ ]:
# ── Create PPO model ──────────────────────────────────────────────────────────

# ═══════════════════════════════════════════════════════════════════════════
# ✏️  TODO 5 — Choose PPO Hyperparameters
# ═══════════════════════════════════════════════════════════════════════════
# PPO has several hyperparameters that control how it learns.
# Fill in reasonable values for the ones marked with ???
# Read the hint for each one carefully before choosing.
#
# Refer back to Section 5 (How PPO Works) if you are unsure.
# ───────────────────────────────────────────────────────────────────────────

model = PPO(
    "MlpPolicy",      # standard MLP policy network — keep this
    train_env,

    # LEARNING RATE — how large each gradient update step is
    # Too high → training is unstable, policy oscillates
    # Too low  → training is very slow
    # Typical range: 1e-4 to 1e-3
    # Hint: 3e-4 is the Adam optimizer default and works well for most RL tasks
    learning_rate = ???,

    # N_STEPS — how many environment steps to collect before each update
    # More steps → more stable gradient estimates, but slower to update
    # Typical range: 1024 to 8192
    # Hint: with 4 parallel envs, 4096 steps means 4096 × 4 = 16384 total transitions per update
    n_steps       = ???,

    batch_size    = 256,    # minibatch size — keep this
    n_epochs      = 10,     # passes over collected data per update — keep this

    # GAMMA — discount factor: how much future rewards matter
    # 0.0 → only immediate reward matters (shortsighted)
    # 1.0 → all future rewards matter equally (no discounting)
    # Hint: for a race car, we care about rewards many steps ahead (completing laps)
    #       so gamma should be close to 1.0
    # Typical range: 0.95 to 0.999
    gamma         = ???,

    gae_lambda    = 0.95,   # GAE smoothing — keep this

    # CLIP_RANGE — the ε in the PPO clip equation
    # Limits how much the policy can change in a single update
    # Too high → policy can change too much, training unstable
    # Too low  → policy changes too slowly, training inefficient
    # From the PPO paper: 0.2 is the recommended default
    clip_range    = ???,

    # ENT_COEF — entropy bonus coefficient
    # Encourages the policy to explore rather than commit too early
    # Too high → policy stays random, never converges
    # Too low  → policy converges too fast to a suboptimal solution
    # Hint: for a task where we want convergence (not pure exploration), keep this small
    # Typical range: 0.0001 to 0.01
    ent_coef      = ???,

    vf_coef         = 0.5,    # value function loss weight — keep this
    max_grad_norm   = 0.5,    # gradient clipping — keep this
    verbose         = 1,
    device          = DEVICE,
    seed            = 42,
    tensorboard_log = "./runs/",
)
# ═══════════════════════════════════════════════════════════════════════════

print()
print("Network architecture:")
print(model.policy)
print()

total_params = sum(p.numel() for p in model.policy.parameters())
print(f"Total trainable parameters: {total_params:,}")
print("(A very small network — intentionally simple for this tutorial)")


Using cuda device

Network architecture:
ActorCriticPolicy(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (pi_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (vf_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (mlp_extractor): MlpExtractor(
    (policy_net): Sequential(
      (0): Linear(in_features=110, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): Tanh()
    )
    (value_net): Sequential(
      (0): Linear(in_features=110, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): Tanh()
    )
  )
  (action_net): Linear(in_features=64, out_features=2, bias=True)
  (value_net): Linear(in_features=64, out_features=1, bias=True)
)

Total trainable parameters: 22,725
(A very small network — intentionally simple for this

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [ ]:
# ── Set up callbacks ──────────────────────────────────────────────────────────
os.makedirs("models/checkpoints", exist_ok=True)
os.makedirs("models/best",        exist_ok=True)

# Save a checkpoint every 100k steps
# Useful if Colab disconnects mid-training
checkpoint_cb = CheckpointCallback(
    save_freq   = 100_000,
    save_path   = "models/checkpoints/",
    name_prefix = f"ppo_{MAP_NAME}",
    verbose     = 1,
)

# Evaluate on clean episodes every 50k steps
# Saves the best model seen so far
eval_cb = EvalCallback(
    eval_env,
    best_model_save_path = "models/best/",
    log_path             = "models/eval_logs/",
    eval_freq            = 50_000,
    n_eval_episodes      = 5,
    deterministic        = True,   # no exploration noise during eval
    render               = False,
    verbose              = 1,
)

# Our custom racing metrics
racing_cb = RacingMetricsCallback(num_envs=NUM_ENVS)

print("Callbacks ready!")
print()
print("What gets saved:")
print("  models/best/best_model.zip         ← best model by eval reward")
print("  models/checkpoints/ppo_*_steps.zip ← snapshots every 100k steps")
print("  models/vecnormalize.pkl            ← reward normalization stats (needed for eval)")

Callbacks ready!

What gets saved:
  models/best/best_model.zip         ← best model by eval reward
  models/checkpoints/ppo_*_steps.zip ← snapshots every 100k steps
  models/vecnormalize.pkl            ← reward normalization stats (needed for eval)


## **---------------------------------------------------**

In [ ]:
# ── TRAIN ─────────────────────────────────────────────────────────────────────


# This is the main training loop.

# SB3 handles everything: rollout collection, advantage computation,
# minibatch updates, logging. We just call .learn().



print("=" * 55)
print("  Starting PPO training")
print(f"  {TOTAL_STEPS:,} steps across {NUM_ENVS} environments")
print(f"  Expected time: ~25–30 minutes on T4 GPU")
print("=" * 55)
print()
print("Metrics to watch in WandB:")
print("  racing/collision_rate  → should drop from 1.0 toward 0.2")
print("  racing/avg_speed       → should rise and stabilize above 4 m/s")
print("  racing/track_progress  → should trend toward 1.0 (full lap)")
print("  rollout/ep_rew_mean    → should trend positive")
print()

model.learn(
    total_timesteps = TOTAL_STEPS,
    callback        = [checkpoint_cb, eval_cb, racing_cb],
    progress_bar    = True,
)

# Save the final model and normalization stats
model.save("models/ppo_part1_final")
train_env.save("models/vecnormalize.pkl")

print()
print("Training complete!")
print("Saved → models/ppo_part1_final.zip")
print("Saved → models/vecnormalize.pkl")

wandb.finish()
train_env.close()

wandb: WARNING When using several event log directories, please call `wandb.tensorboard.patch(root_logdir="...")` before `wandb.init`


  Starting PPO training
  1,000,000 steps across 4 environments
  Expected time: ~25–30 minutes on T4 GPU

Metrics to watch in WandB:
  racing/collision_rate  → should drop from 1.0 toward 0.2
  racing/avg_speed       → should rise and stabilize above 4 m/s
  racing/track_progress  → should trend toward 1.0 (full lap)
  rollout/ep_rew_mean    → should trend positive

Logging to ./runs/PPO_3


Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

-------------------------------------
| perf/                  |          |
|    fps_per_env         | 0        |
| racing/                |          |
|    avg_min_scan        | 0.52     |
|    avg_speed           | 2.88     |
|    collision_rate      | 1        |
|    lap_completion_rate | 0        |
|    track_progress      | 0.0236   |
| rollout/               |          |
|    ep_len_mean         | 161      |
|    ep_len_std          | 127      |
|    ep_rew_mean         | -27.7    |
| time/                  |          |
|    fps                 | 724      |
|    iterations          | 1        |
|    time_elapsed        | 22       |
|    total_timesteps     | 16384    |
-------------------------------------
-----------------------------------------
| perf/                   |             |
|    fps_per_env          | 0           |
| racing/                 |             |
|    avg_min_scan         | 0.335       |
|    avg_speed            | 2.22        |
|    collision_rate       

Eval num_timesteps=200000, episode_reward=238.02 +/- 1.31

... (22 outputs left out)

New best mean reward!

-------------------------------------
| perf/                  |          |
|    fps_per_env         | 0        |
| racing/                |          |
|    avg_min_scan        | 0.659    |
|    avg_speed           | 5.89     |
|    collision_rate      | 0.774    |
|    lap_completion_rate | 0.303    |
|    track_progress      | 0.776    |
| rollout/               |          |
|    ep_len_mean         | 8.59e+03 |
|    ep_len_std          | 5.44e+03 |
|    ep_rew_mean         | 239      |
| time/                  |          |
|    fps                 | 487      |
|    iterations          | 62       |
|    time_elapsed        | 2082     |
|    total_timesteps     | 1015808  |
-------------------------------------



Training complete!
Saved → models/ppo_part1_final.zip
Saved → models/vecnormalize.pkl


eval/mean_ep_length,▁▁▁▁▁
eval/mean_reward,▁▇███
global_step,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇███
perf/fps_per_env,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
racing/avg_min_scan,▁▄▂▄▂▃▃▂▂▅▁▃▁▃▄▅▄▇▃▂▅▄▄▅▆▇▇█▆▆▅▅▅▅▅▄▄▄▅▄
racing/avg_speed,▃▂▁▃▂▃▂▂▂▂▁▃▃▂▃▃▃▃▂▄▃▃▃▃▃▄▆▆▆▆▇▇████████
racing/collision_rate,██████████████████████████████▇▇▇▆▆▃▃▂▂▁
racing/lap_completion_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▃▃▃▅▅▆▇▇▇████
racing/track_progress,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▄█▆▂█▄▄▅▆▆▆▆▆▆▇▇▇▇▇▇
rollout/ep_len_mean,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▆▆▆▆▆▇▇▇▇▇▇▇████
+13,...


## **---------------------------------------------------**

## 6.2 Save Your Trained Models

Run the cell below to download all your saved models as a single zip file.

The archive includes:
- `models/best/best_model.zip` — best policy found during training
- `models/ppo_part1_final.zip` — final model at end of training  
- `models/vecnormalize.pkl` — reward normalization stats (**required** alongside any model file to run eval correctly)
- `models/checkpoints/` — snapshots saved every 100k steps

> **Running locally:** If you have the F1TENTH gym and virtual environment set up on your local machine, downloading this model archive will let you run the trained model eval locally to see the car driving with a live render window.
_**However, for this version, we will save a headless recording directly in the notebook using rgb_array rendering, so you can visualize your results without any local setup required.**_

> _It will also be useful for the actual deployement on the car which we shall cover in the notebooks later._

In [ ]:
!zip -r models_archive.zip /content/models

updating: content/models/ (stored 0%)
updating: content/models/vecnormalize.pkl (deflated 43%)
updating: content/models/best/ (stored 0%)
updating: content/models/best/best_model.zip (stored 0%)
updating: content/models/ppo_part1_final.zip (stored 0%)
updating: content/models/checkpoints/ (stored 0%)
updating: content/models/checkpoints/ppo_Spielberg_40000_steps.zip (stored 0%)
updating: content/models/checkpoints/ppo_Spielberg_80000_steps.zip (stored 0%)
updating: content/models/eval_logs/ (stored 0%)
updating: content/models/eval_logs/evaluations.npz (deflated 55%)


In [ ]:
# Download the .zip folder locally

from google.colab import files
files.download('models_archive.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

# **Section 7 — Monitoring & Metrics**

## 7.1 Understanding the Training Metrics

Now that training has finished (or is running), let's understand what all those numbers mean. *Run Section 8 and compare your training metrics with these values to analyze the training quality*.

### ✅ What Healthy Training Looks Like

| Metric | What It Measures | Healthy Trend |
|---|---|---|
| `racing/collision_rate` | Fraction of episodes ending in crash | 1.0 → below 0.3 |
| `racing/avg_speed` | Average speed during episode | Rising, stable above 4 m/s |
| `racing/track_progress` | Distance traveled (0→1 per lap) | Trending toward 1.0 |
| `racing/lap_completion_rate` | Fraction completing a full lap | Rising toward 0.7+ |
| `rollout/ep_rew_mean` | Average total episode reward | Trending positive |
| `rollout/ep_len_std` | Std of episode lengths | Decreasing = more consistent |
| `train/std` | Policy action randomness | Stable, not rising |
| `train/explained_variance` | Value function accuracy | Trending toward 0.8+ |
| `perf/fps_per_env` | Simulator speed per env | Should be ~500–1000 |

### ⚠️ Warning Signs

| Warning Sign | What It Means | What To Do |
|---|---|---|
| `train/std` keeps rising | Policy becoming more random | Lower `ent_coef` |
| `avg_speed` drops toward 0 | Car learned to crawl slowly | Add speed reward |
| `collision_rate` plateaus above 0.5 | Not learning fast enough | More steps, or tune LR |
| `explained_variance` stays near 0 | Value function not fitting | Check reward scale |
| `ep_len_std` very high | Inconsistent policy | More training needed |

---

# **Section 8 — Evaluation**

## 8.1 Evaluate the Trained Policy

Training is done. Let's see how well the policy actually drives.

In [ ]:
# ── Load and evaluate ─────────────────────────────────────────────────────────

# Load the best model found during training (by eval reward)
# If you want the final model instead: "models/ppo_part1_final"
model_path = "models/best/best_model"
print(f"Loading model: {model_path}")

model = PPO.load(model_path, device="cpu")

# Create a fresh environment for evaluation
# Note: during eval we use deterministic=True — no exploration noise
env = F1TenthRLWrapper(map_name=MAP_NAME)

NUM_EVAL_EPISODES = 5
all_rewards  = []
all_speeds   = []
all_progress = []
all_laps     = []

print()
print(f"Running {NUM_EVAL_EPISODES} evaluation episodes...")
print("─" * 65)

for ep in range(NUM_EVAL_EPISODES):
    obs, _       = env.reset()
    total_reward = 0.0
    done         = False
    steps        = 0
    speeds       = []
    laps         = 0

    while not done:
        # deterministic=True: always pick the most likely action (no sampling)
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        done          = terminated or truncated
        steps        += 1
        speeds.append(info.get("speed", 0.0))
        if info.get("lap_completed", False):
            laps += 1

    avg_spd  = np.mean(speeds)
    progress = info.get("progress", 0)
    crashed  = info.get("collision", False)

    all_rewards.append(total_reward)
    all_speeds.append(avg_spd)
    all_progress.append(progress)
    all_laps.append(laps)

    status = "CRASH" if crashed else ("TIMEOUT" if steps >= 10000 else "DONE")
    print(f"  Ep {ep+1}: {status} | "
          f"reward={total_reward:7.1f} | "
          f"steps={steps:5d} | "
          f"speed={avg_spd:.2f} m/s | "
          f"progress={progress:.3f} | "
          f"laps={laps}")

print("─" * 65)
print(f"  Average reward:   {np.mean(all_rewards):.1f} ± {np.std(all_rewards):.1f}")
print(f"  Average speed:    {np.mean(all_speeds):.2f} m/s")
print(f"  Average progress: {np.mean(all_progress):.3f} (1.0 = full lap)")
print(f"  Total laps:       {sum(all_laps)}")

env.close()

Loading model: models/best/best_model


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[ENV] Loaded 1692 waypoints from track

Running 5 evaluation episodes...
─────────────────────────────────────────────────────────────────
  Ep 1: TIMEOUT | reward=  316.2 | steps=10000 | speed=5.97 m/s | progress=0.785 | laps=1
  Ep 2: TIMEOUT | reward=  316.2 | steps=10000 | speed=5.97 m/s | progress=0.778 | laps=1
  Ep 3: TIMEOUT | reward=  316.2 | steps=10000 | speed=5.97 m/s | progress=0.783 | laps=1
  Ep 4: TIMEOUT | reward=  316.1 | steps=10000 | speed=5.97 m/s | progress=0.777 | laps=1
  Ep 5: TIMEOUT | reward=  316.3 | steps=10000 | speed=5.97 m/s | progress=0.784 | laps=1
─────────────────────────────────────────────────────────────────
  Average reward:   316.2 ± 0.0
  Average speed:    5.97 m/s
  Average progress: 0.782 (1.0 = full lap)
  Total laps:       5


---

## 8.2 Record Eval Video in Colab - _See your Policy in Action !_

Since our colab session has no display, we can't open a render window directly. Instead we create a fresh environment with `render_mode="rgb_array"` which returns frames as numpy arrays — no display needed. We wrap it with `DummyVecEnv` and load the saved `VecNormalize` stats so the policy sees the same observation scale it was trained on, then capture every 5th frame and save as an MP4.


In [ ]:
# ── Record eval video — Using F1TENTH simulator rendering ───────────────────
!pip install -q imageio imageio-ffmpeg

import imageio

# Load model and environment with rendering
print("Loading model and environment...")
model = PPO.load("models/best/best_model", device="cpu")

# Create environment with rgb_array rendering
vec_env = DummyVecEnv([lambda: F1TenthRLWrapper(map_name=MAP_NAME, render_mode="rgb_array")])
vec_env = VecNormalize.load("models/vecnormalize.pkl", vec_env)
vec_env.training = False
vec_env.norm_reward = False

# Record episode
print("Recording episode...")
frames = []
obs = vec_env.reset()
done = False
step = 0
max_steps = 10000

while not done and step < max_steps:
    action, _ = model.predict(obs, deterministic=True)
    obs, _, dones, info = vec_env.step(action)
    done = dones[0]
    step += 1

    # Capture every 5th frame
    if step % 5 == 0:
        frame = vec_env.envs[0].render()
        if frame is not None:
            frames.append(frame)

vec_env.close()

if frames:
    print(f"✅ Captured {len(frames)} frames from {step} steps")
    imageio.mimsave('eval_render.mp4', frames, fps=20)
    print("✅ Saved → eval_render.mp4")

    from google.colab import files
    files.download('eval_render.mp4')
else:
    print("❌ No frames captured. Check if the environment renders correctly.")

Loading model and environment...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/pyqtgraph/parametertree/Parameter.py:140: DeprecationWarning: Parameter has no default value. Pass a default, or use setDefault(). This will no longer set an implicit default after January 2025.
  return cls(**opts)


[ENV] Loaded 1692 waypoints from track
Recording episode...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

✅ Captured 2000 frames from 10000 steps
✅ Saved → eval_render.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

# **Section 9 — Summary & Next Steps**

## 9.1 What We Built

In about 5 minutes of training, we went from a completely random policy that crashes immediately to a car that:
- Drives forward at 4–6 m/s (2-3 m/s without speed reward term)
- Avoids walls most of the time
- Completes full laps on the Spielberg track

All without any hand-coded rules — purely from the reward signal.

## 9.2 What's Still Wrong

If you watched the evaluation output, you probably noticed:

- **Inconsistent performance** — some episodes complete laps, some crash early
- **Squiggly steering** — no penalty for jerky inputs
- **Not necessarily fast** — the speed bonus was small, so it might take a conservative line
- **No awareness of the racing line** — the car goes forward, but not optimally

These are all reward function problems, not algorithm problems. PPO is doing its job — it's optimizing exactly what we asked it to. We just didn't ask for quite enough.

## 9.3 Exercises — Try These Before Part 2

1. **Change the crash penalty** from -30 to -5. Does the car take more risks? Does it crash more?
2. **Remove the speed reward** entirely (`reward += 0` instead of `0.03 * ...`). How does avg_speed change?
3. **Change NUM_ENVS from 4 to 1**. How much slower is training? How does it affect quality?

---

## 9.4 Saving the Model Files - IMPORTANT for local evaluation, and deployement on the car

> **💾 Saving your model:** If you want to keep the trained model, download `models/ppo_part1_final.zip` and `models/vecnormalize.pkl` from the Colab file browser on the left. You will need both files to load and run the policy later.

## 9.5 Save Models to Google Drive *(Optional)*

Colab sessions are temporary — models are lost when the session ends. Uncomment the cell below to save your model to Google Drive.

In [ ]:
# ── Optional: Save models to Google Drive ─────────────────────────────────────
# Uncomment and run this cell to save your trained model to Google Drive
#so it persists after the Colab session ends.

#from google.colab import drive
#drive.mount('/content/drive')

#import shutil
#shutil.copy('models/ppo_part1_final.zip', '/content/drive/MyDrive/')
#shutil.copy('models/vecnormalize.pkl',    '/content/drive/MyDrive/')
#shutil.copy('models/best/best_model.zip', '/content/drive/MyDrive/')
#print("Models saved to Google Drive!")

#print("Uncomment the lines above to save your model to Google Drive.")

# **Section 10 - Video Demo**

Below is sample of how your car would look in the simulation:

**🖥️ Local Render** — Full simulator visualization running on a local machine:
https://drive.google.com/file/d/1r8cnzX65qW-yqa8CoUi1e9ji8fHsQA2e/view?usp=sharing

**☁️ Colab Headless Recording** — The rgb_array capture from this notebook, no local setup required:
https://drive.google.com/file/d/1VtQlQb_8VuquyW-A9J84_nnre3fIX1P5/view?usp=sharing

# **Section 11 - Coming Up in Part 2**

> We will fix the problems above by improving the reward function:

| Problem | Part 2 Fix |
|---|---|
| Squiggly steering | Steering smoothness penalty |
| Hugging walls | Wall proximity penalty |
| No lap completion signal | Lap completion bonus (+50) |
| Slow in corners | Frenet-frame based reward |

By the end of Part 2, you will have a refined reward function that will produce smoother, faster, and more consistent driving, and cover the key techniques needed to deploy your trained policy onto the physical car for racing.